# CircuitVision • Unified YOLO11 Training (Round 2 + Round 3 + Sam)

**Goal: ONE student-trained YOLO11n physical-object detector, not three separate CPU models.**

Training reuses **Round 2's trusted YOLO11n checkpoint**, the original two datasets, Round 3's two additional physical-component datasets, and **Sam's original annotated data.zip**.

The unified target has **87 categories**: the Round 3 physical list of 81 (with all original 57 IDs preserved) plus 6 specialist visual subtypes: Ceramic Capacitor, LED Blue, LED Green, Resistor 10K, Resistor 2K, Resistor 52K. The remaining four Sam labels map to existing broad categories. Sam's checkpoint is **not weight-averaged** with Round 2/3 because YOLO11s and YOLO11n differ architecturally; the learning from Sam is incorporated via his **annotated images**.

**Training is performed in three staged fine-tuning passes.** At each stage, the best candidate is selected only by a fixed validation set. The separate test set is evaluated **once at the end**. Repeated training does not guarantee improvement and may cause overfitting.

**Sam's data.zip is REQUIRED.** The three files previously sent (my_model.pt, yolo_detector.py, YOLO Notebook.ipynb) are not the labeled dataset. This notebook halts before expensive dataset downloads if data.zip is absent.

**Important:** Even after this training, a predicted “Resistor 10K” is a visual label, **not a measured 10 kΩ resistance**. Extensive varied real-world testing is required before trusting exact component values. The working production website and Round 2 model are not modified by this notebook.

## 1 — Setup and verify GPU

Select **Runtime → Change runtime type → T4 GPU**. Press **Run all** only after you have Sam's labeled dataset ZIP. A GPU is required for training. The notebook stops if CUDA is unavailable.

In [ ]:
%pip -q install "ultralytics>=8.3,<9" "roboflow>=1.1" "PyYAML>=6" "onnx>=1.15" "onnxruntime>=1.18" "onnxslim>=0.1"
import torch, pathlib, json, os, gc, hashlib, io, zipfile, shutil, random, re, math
from pathlib import Path
from getpass import getpass
from collections import Counter
import yaml
print('GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'MISSING')
assert torch.cuda.is_available(),'Please enable T4 GPU under Runtime settings before training.'

## 1b — Locate Sam’s original annotated dataset **before** expensive downloads

The notebook first checks the original `MyDrive/YOLO/dataset/data.zip` location in Google Drive. If absent, Colab prompts for the actual `data.zip`. Uploading only `my_model.pt` or your Colab notebook is **not enough**.

In [ ]:
# Locate Sam's original annotated data.zip FIRST, before downloading other sources.
from pathlib import Path
import shutil, zipfile
sam_data_archive=Path('/content/Sam_Original_Annotated_Dataset.zip')
candidates=[
  Path('/content/drive/MyDrive/YOLO/dataset/data.zip'),
  Path('/content/gdrive/MyDrive/YOLO/dataset/data.zip'),
  Path('/content/data.zip'),
  sam_data_archive,
]
existing=next((p for p in candidates if p.is_file()),None)
if existing is None:
    from google.colab import drive
    try:
        drive.mount('/content/drive')
    except Exception as e:
        print('Google Drive mounting was not completed:',str(e)[:200])
    existing=next((p for p in candidates if p.is_file()),None)
if existing is None:
    print('Sam dataset not found on Google Drive. Upload the ORIGINAL labeled data.zip now.')
    from google.colab import files
    uploaded=files.upload()
    if len(uploaded)==1:
        existing=Path('/content')/next(iter(uploaded))
assert existing is not None and existing.is_file(), (
    'Sam labeled data.zip is required to train all three datasets. '
    'Find MyDrive/YOLO/dataset/data.zip or upload it, not my_model.pt.')
assert zipfile.is_zipfile(existing),'Sam dataset must be a valid ZIP of labeled images'
if existing.resolve()!=sam_data_archive.resolve():
    shutil.copy2(existing,sam_data_archive)
print('Sam dataset supplied:',sam_data_archive,'size:',sam_data_archive.stat().st_size,'bytes')

## 2 — Get the audited physical-component datasets

If you stayed in the same Colab runtime as the audit, existing files under /content/circuitvision_dataset_audit are reused. Otherwise, enter your Roboflow key privately. Download failures stop training—no silent data gaps.

In [ ]:
from roboflow import Roboflow
SOURCES=[
 dict(name='eesob_original',workspace='android-yolo',project='eesob',version=4,required=True),
 dict(name='electronics_original',workspace='vishnupriyas-workspace',project='electronics-components-sveck',version=7,required=True),
 dict(name='electronic_component_detection',workspace='personal-8hq68',project='electronic-component-detection-m6sop',version=3,required=True),
 dict(name='lab_instruments',workspace='myspace-natcf',project='deteksi-alat-elektronik',version=2,required=True),
]
DATA_ROOT=Path('/content/circuitvision_dataset_audit')
DATA_ROOT.mkdir(parents=True,exist_ok=True)
roots={}
rf=None
for source in SOURCES:
    folder=DATA_ROOT/source['name']
    existing=next(iter(folder.rglob('data.yaml')),None) if folder.exists() else None
    if existing is not None:
        roots[source['name']]=existing.parent
        print('Reused:',source['name'])
        continue
    if rf is None:
        api_key=getpass('Roboflow API key (hidden): ').strip()
        assert api_key,'Roboflow API key required'
        rf=Roboflow(api_key=api_key)
        del api_key
    problems=[]
    for fmt in ('yolov11','yolov8'):
        try:
            record=rf.workspace(source['workspace']).project(source['project']).version(source['version']).download(fmt,location=str(folder))
            roots[source['name']]=Path(record.location)
            print('Downloaded:',source['name'],fmt)
            break
        except Exception as exc:
            problems.append(fmt+': '+str(exc)[:250])
    if source['name'] not in roots:
        raise RuntimeError('Cannot download '+source['name']+': '+'; '.join(problems))
del rf
assert len(roots)==4
print('All four physical-component sources available')

## 3 — Load and verify your **real Round 2** checkpoint

Downloads the exact currently deployed Round 2 training ZIP from your GitHub repository and checks its pinned SHA256. The class list in the PyTorch checkpoint must agree with classes.json.

In [ ]:
from urllib.request import urlopen,Request
from ultralytics import YOLO
MODEL_URL='https://raw.githubusercontent.com/aquinogeraldedmiranda-art/CircuitVision-YOLO/main/models/CircuitVision_Accuracy_Round2.zip'
PINNED_SHA256='40b97bd83288d8dcc15082019ff335612683cd09d90997ca55ff27ca5d03cae3'
WEIGHTS_DIR=Path('/content/circuitvision_round2_verified')
WEIGHTS_DIR.mkdir(exist_ok=True)
MODEL_PT=WEIGHTS_DIR/'best.pt'
if not MODEL_PT.exists():
    raw=urlopen(Request(MODEL_URL,headers={'User-Agent':'CircuitVision-Round3-Colab'}),timeout=180).read(25_000_000)
    digest=hashlib.sha256(raw).hexdigest()
    assert digest==PINNED_SHA256,f'Wrong Round 2 archive SHA: {digest}'
    with zipfile.ZipFile(io.BytesIO(raw)) as z:
        assert 'best.pt' in z.namelist() and 'classes.json' in z.namelist()
        MODEL_PT.write_bytes(z.read('best.pt'))
        (WEIGHTS_DIR/'classes.json').write_bytes(z.read('classes.json'))
    del raw
baseline=YOLO(str(MODEL_PT))
old_names=[str(baseline.names[i]) for i in range(len(baseline.names))]
assert len(old_names)==57,'Round 2 model must contain 57 prior categories'
original_map=json.loads((WEIGHTS_DIR/'classes.json').read_text())
assert [original_map[str(i)] for i in range(57)]==old_names,'Checkpoint class mapping differs from saved classes.json'
print('Verified original 57 classes, preserving exact index order')
print(old_names)

## 4 — Define an explicit, defensible class-name mapping

Parts that cannot be distinguished visually (e.g. capacitor printed values) are merged into their broad category rather than given a false numeric value. New instrument categories are added. Every source class must be mapped; unexpected spelling stops the merge.

In [ ]:
def norm(s):
    return re.sub(r'\s+',' ',re.sub(r'[_-]+',' ',str(s).strip().lower())).strip()
existing={norm(n):n for n in old_names}
assert len(existing)==57
EXTRA_ALIASES={
 '1-5-Volt-Battery':'Battery',
 '3-3-Volt-Battery':'Battery',
 '7-Segment-Display':'7-segment',
 '9-Volt-Battery':'Battery 9V',
 'Arduino-Mega':'Arduino Mega',
 'Arduino-Nano':'Arduino Nano',
 'Arduino-Uno':'Arduino Uno',
 'BJT-Transistor':'Transistor',
 'Bluetooth-Module':'Bluetooth',
 'Breadboard':'Breadboard',
 'Bridge-Rectifier':'Bridge Rectifier',
 'Buck-Converter':'Power Module',
 'Buzzer':'Buzzer',
 'Capacitor-10mf':'Capacitor',
 'Capacitor-470mf':'Capacitor',
 'DC-Motor':'DC Motor',
 'Diode':'Diode',
 'ESP32':'ESP32',
 'ESP32-CAM':'ESP32 CAM',
 'FT-232-USB-Serial-Module':'USB Serial Module',
 'Film-Capacitor':'Capacitor',
 'Fuse':'Fuse',
 'Fuse-Base':'Fuse Holder',
 'GSM-Module':'GSM Module',
 'Gas-Sensor':'Gaz',
 'Heat-Sink':'Heat Sink',
 'High-Voltage-Ceramic-Capacitor':'Capacitor',
 'Humidity-Sensor':'Humidity Sensor',
 'IC-Base-14-Pin':'IC Socket',
 'IC-Base-28-Pin':'IC Socket',
 'IC-Chip':'IC',
 'IGBT':'IGBT',
 'IR-Sensor':'IR Sensor',
 'Inductor':'Inductor',
 'Keypad':'Keypad',
 'LCD-Display':'LCD',
 'LDR-Sensor':'LDR Sensor',
 'LED-Light':'LED',
 'Low-Voltage-Ceramic-Capacitor':'Capacitor',
 'MLC-Capacitor':'Capacitor',
 'MOSFET':'MOSFET',
 'Motion-Sensor':'PIR',
 'Motor-Driver':'H Bridge',
 'NTC-Thermistor':'Thermistor',
 'OLED-Display':'LCD OLED',
 'Pin-Header':'Pin Header',
 'Push-Switch':'Button',
 'RFID-Scanner':'RFID',
 'Raindrops-Module':'Rain Sensor',
 'Relay-Module':'Relai',
 'Resistor':'Resistor',
 'Rocker-Switch':'Switch',
 'Servo-Motor':'Servo',
 'Soil-Moisture-Sensor':'Soil Moisture',
 'Sonar-Sensor':'Ultrasonic',
 'TCRT5000':'IR Sensor',
 'Tact-Switch':'Button',
 'Taper-Potentiometer':'Potentiometer',
 'Trimmer-Potentiometer':'Potentiometer',
 'Water-Sensor':'Water Sensor',
 'Zener-Diode':'Diode',
}
INSTRUMENT_ALIASES={
 'breadboard':'Breadboard',
 'component box':'Component Storage Box',
 'digital oscilloscope':'Oscilloscope',
 'function generator':'Function Generator',
 'multimeter':'Multimeter',
 'power strip':'Power Strip',
 'trainer kit':'Electronics Trainer Kit',
}
def resolve(source,original_name):
    if source in ('eesob_original','electronics_original'):
        key=norm(original_name)
        assert key in existing,f'Original class absent from Round 2: {source} / {original_name}'
        return existing[key]
    entries=EXTRA_ALIASES if source=='electronic_component_detection' else INSTRUMENT_ALIASES
    assert original_name in entries,f'NEW CLASS needs manual review: {source} / {original_name}'
    proposed=entries[original_name]
    return existing.get(norm(proposed),proposed)
print('Mapping plan:',len(EXTRA_ALIASES),'expanded part classes,',len(INSTRUMENT_ALIASES),'lab-instrument classes')

## 5 — Load the validated Round 2 YOLO merge reader

Reuses your project's original class-ID parsing and polygon-to-box conversion logic instead of guessing source IDs. This step downloads our public notebook, not any private data.

In [ ]:
import json
round2_notebook_url='https://raw.githubusercontent.com/aquinogeraldedmiranda-art/CircuitVision-YOLO/main/notebooks/CircuitVision_Accuracy_Round2.ipynb'
with urlopen(Request(round2_notebook_url,headers={'User-Agent':'CircuitVision-Training'}),timeout=60) as stream:
    original_nb=json.load(stream)
reader=''.join(original_nb['cells'][7]['source'])
assert all(n in reader for n in ('def read_rows(', 'def source_samples(', 'def read_class_names('))
exec(compile(reader,'round2_merge_reader.py','exec'),globals())
source_data={}
for s in SOURCES:
    name=s['name']
    samples,classes=source_samples(Path(roots[name]),name)
    local_mapped=[resolve(name,c) for c in classes]
    source_data[name]=(samples,classes,local_mapped)
    print(name,':',len(classes),'source classes;','train samples:',len(samples['train']),
          'mapped examples:',dict(list(zip(classes,local_mapped))[:6]))
expanded=sorted({canonical for _,_,mapped in source_data.values() for canonical in mapped if canonical not in old_names})
class_names=old_names+expanded
assert len(set(class_names))==len(class_names),'Duplicate canonical names'
assert class_names[:57]==old_names,'Original indices must not change'
print('Round 3 proposed categories:',len(class_names))
print('New categories:',expanded)

## 5b — Integrate Sam’s **labeled** 10-class dataset

The original class-number order is checked against Sam’s verified checkpoint. Images and YOLO labels are paired and checked; exact duplicate images are removed. We reserve a small test split where possible. Class-specific values remain **unverified visual labels**, not electrical measurements. This is a strict preflight: never merge different class IDs by accident.

In [ ]:
# Sam's notebook originally used MyDrive/YOLO/dataset/data.zip and
# a flat image + YOLO annotation collection with classes.txt.
# Also accept image/labels folders and some pre-split exports.
import zipfile,hashlib,random,os,stat
from pathlib import PurePosixPath
SAM_SOURCE=Path('/content/circuitvision_sam_source')
SAM_READY=Path('/content/circuitvision_sam_ready')
if SAM_SOURCE.exists():shutil.rmtree(SAM_SOURCE)
SAM_SOURCE.mkdir(parents=True)
max_bytes=2_500_000_000
total_bytes=0
with zipfile.ZipFile(sam_data_archive) as archive:
    for item in archive.infolist():
        p=PurePosixPath(item.filename)
        if p.is_absolute() or '..' in p.parts or '\\' in item.filename:
            raise RuntimeError('Unsafe path in Sam dataset archive')
        if (item.external_attr>>16)&0o170000 == stat.S_IFLNK:
            raise RuntimeError('Unsafe symlink in dataset ZIP')
        total_bytes+=item.file_size
        if total_bytes>max_bytes:raise RuntimeError('Dataset exceeds 2.5 GB safe limit')
    archive.extractall(SAM_SOURCE)
SAM_CLASSES=['BUZZER','CERAMIC CAPACITOR','DIODE','LED (BLUE)','LED (GREEN)',
             'POTENTIOMETER','POWER MODULE','RESISTOR 10K','RESISTOR 2K','RESISTOR 52K']
txts=[p for p in SAM_SOURCE.rglob('*') if p.is_file() and p.name.lower()=='classes.txt']
configs=[p for p in SAM_SOURCE.rglob('data.yaml') if p.is_file()]
if txts:
    raw_names=[s.strip() for s in txts[0].read_text(encoding='utf8-sig').splitlines() if s.strip()]
elif len(configs)==1:
    raw_names=read_class_names(configs[0].parent)
else:
    raise ValueError('Sam data.zip must contain classes.txt or one data.yaml')
assert [norm(x) for x in raw_names]==[norm(x) for x in SAM_CLASSES], (
    'Sam dataset class IDs disagree with the 10-class checkpoint. '
    'Do not guess or train on a mismatched class list: '+str(raw_names))
images=[]
miss=0
for path in SAM_SOURCE.rglob('*'):
    if not path.is_file() or path.suffix.lower() not in IMAGE_EXTS:continue
    ann=path.with_suffix('.txt')
    if not ann.is_file() and path.parent.name=='images':
        ann=path.parent.parent/'labels'/(path.stem+'.txt')
    if not ann.is_file():
        miss+=1
        continue
    rows=read_rows(ann,len(SAM_CLASSES))
    if not rows:continue
    images.append((path,ann,{cid for cid,_ in rows}))
print('Sam image/label pairs:',len(images),'unlabeled images skipped:',miss)
assert len(images)>=30, (
    'Fewer than 30 labeled images. Inspect data.zip; it may not be the original Sam training set.')
# Deduplicate exact copies before splitting; source augmentations may still
# produce near-duplicates, so these metrics are only preliminary.
rng=random.Random(59)
rng.shuffle(images)
unique={}
for img,ann,ids_present in images:
    digest=hashlib.sha256(img.read_bytes()).hexdigest()
    if digest not in unique:unique[digest]=(img,ann,ids_present)
images=list(unique.values())
assert len(images)>=30,'Insufficient unique Sam images'
# Seed one training example per class, then split remaining into train/val/test.
train=[];rest=[];covered=set()
for candidate in images:
    unseen=candidate[2]-covered
    if unseen:
        train.append(candidate);covered.update(candidate[2])
    else:rest.append(candidate)
assert covered==set(range(10)), 'Some Sam component classes have no labeled examples'
rng.shuffle(rest)
test_n=max(4,int(len(rest)*0.13))
val_n=max(4,int(len(rest)*0.13))
test=rest[:test_n];valid=rest[test_n:test_n+val_n];train+=rest[test_n+val_n:]
assert train and valid and test, 'Sam dataset cannot form independent train/val/test'
if SAM_READY.exists():shutil.rmtree(SAM_READY)
for split,items in [('train',train),('val',valid),('test',test)]:
    for img,ann,_ in items:
        key=hashlib.sha256(img.read_bytes()).hexdigest()[:20]
        dest_img=SAM_READY/split/'images'/(key+img.suffix.lower())
        dest_lbl=SAM_READY/split/'labels'/(key+'.txt')
        dest_img.parent.mkdir(parents=True,exist_ok=True)
        dest_lbl.parent.mkdir(parents=True,exist_ok=True)
        shutil.copy2(img,dest_img);shutil.copy2(ann,dest_lbl)
(SAM_READY/'data.yaml').write_text(yaml.safe_dump({
 'path':str(SAM_READY),'train':'train/images','val':'val/images',
 'test':'test/images','names':SAM_CLASSES,'nc':10},sort_keys=False))
SAM_MAP={
 'BUZZER':'Buzzer','CERAMIC CAPACITOR':'Ceramic Capacitor',
 'DIODE':'Diode','LED (BLUE)':'LED Blue','LED (GREEN)':'LED Green',
 'POTENTIOMETER':'Potentiometer','POWER MODULE':'Power Module',
 'RESISTOR 10K':'Resistor 10K','RESISTOR 2K':'Resistor 2K',
 'RESISTOR 52K':'Resistor 52K'
}
SAM_SUBCLASSES=['Ceramic Capacitor','LED Blue','LED Green',
                'Resistor 10K','Resistor 2K','Resistor 52K']
sam_samples,sam_classes=source_samples(SAM_READY,'sam_specialist')
sam_mapped=[SAM_MAP[s] for s in sam_classes]
source_data['sam_specialist']=(sam_samples,sam_classes,sam_mapped)
SOURCES.append(dict(name='sam_specialist',workspace='private-sam',project='local-annotated-data',version='local'))
class_names+= [s for s in SAM_SUBCLASSES if s not in class_names]
assert class_names[:57]==old_names
assert len(class_names)==87, 'Unexpected unified class count; inspect overlapping names'
train_instances=Counter()
for photo in sam_samples['train']:
    for local_id,_ in read_rows(photo.labels,len(sam_classes)):
        train_instances[sam_classes[local_id]]+=1
print('Sam train/validation/test image counts:',len(train),len(valid),len(test))
print('Sam training box counts per specialized class:',dict(train_instances))
if any(n<6 for n in train_instances.values()) or len(train_instances)<10:
    print('WARNING: Some Sam categories have very few labeled training boxes; '
          'exact-value accuracy cannot be trusted without more diverse data')
print('Physical classes:',len(class_names)-len(SAM_SUBCLASSES),
      'specialist subtypes:',len(SAM_SUBCLASSES),'TOTAL:',len(class_names))

## 6 — Merge sampled images and relabel every box

Source-aware selection protects smaller instrument classes; per-source limits avoid exhausting the free Colab session. **Caution:** Augmented near-duplicates may cross provider splits; this script only removes byte-identical duplicates. Audit dataset origin and leakage before publishing scientific evaluation results.

In [ ]:
from collections import defaultdict
# Per SOURCE sampling limits; change these only if GPU time and disk space allow.
SPLIT_LIMITS={
 'eesob_original':{'train':3200,'val':250,'test':200},
 'electronics_original':{'train':3200,'val':250,'test':200},
 'electronic_component_detection':{'train':2400,'val':250,'test':200},
 'lab_instruments':{'train':1000,'val':100,'test':48},
 'sam_specialist':{'train':99999,'val':99999,'test':99999},
}
MERGED=Path('/content/circuitvision_round3_merged')
if MERGED.exists():shutil.rmtree(MERGED)
ids={n:i for i,n in enumerate(class_names)}
images_c=Counter();boxes_c=Counter();duplicates=0;missing_annotations=0
rng=random.Random(47);all_hashes=set()
# Test and validation are reserved BEFORE selecting training examples.
for split in ('test','val','train'):
 for spec in SOURCES:
    source=spec['name']
    samples,originals,mapped=source_data[source]
    pool=samples[split].copy()
    rng.shuffle(pool)
    cap=SPLIT_LIMITS[source][split]
    if split=='train':
        # Ensure early coverage of as many labels as the source contains.
        by_class=defaultdict(list)
        for photo in pool:
            for cls,_ in read_rows(photo.labels,len(originals)):
                if len(by_class[cls])<15:by_class[cls].append(photo)
        priority=[];seen_paths=set()
        for class_id in sorted(by_class):
            for photo in by_class[class_id]:
                if photo.path not in seen_paths:
                    priority.append(photo);seen_paths.add(photo.path)
        pool=(priority+[p for p in pool if p.path not in seen_paths])[:cap]
    else:
        pool=pool[:cap]
    for sample in pool:
        digest=hashlib.sha256(sample.path.read_bytes()).hexdigest()
        if digest in all_hashes:
            duplicates+=1;continue
        all_hashes.add(digest)
        annotations=read_rows(sample.labels,len(originals))
        if not sample.labels.exists():missing_annotations+=1
        filename=source+'_'+digest[:18]+sample.path.suffix.lower()
        out_image=MERGED/split/'images'/filename
        out_label=MERGED/split/'labels'/(Path(filename).stem+'.txt')
        out_image.parent.mkdir(parents=True,exist_ok=True)
        out_label.parent.mkdir(parents=True,exist_ok=True)
        shutil.copy2(sample.path,out_image)
        with out_label.open('w') as dest:
            for class_id,xywh in annotations:
                new_name=mapped[class_id]
                dest.write(str(ids[new_name])+' '+' '.join(f'{f:.6f}' for f in xywh)+'\n')
                boxes_c[(split,new_name)]+=1
        images_c[(split,source)]+=1
assert sum(v for (sp,_),v in images_c.items() if sp=='train')>3000
assert sum(v for (sp,_),v in images_c.items() if sp=='test')>300
output_config={'path':str(MERGED.resolve()),'train':'train/images','val':'val/images','test':'test/images','names':class_names,'nc':len(class_names)}
(MERGED/'data.yaml').write_text(yaml.safe_dump(output_config,sort_keys=False))
print('Images by source and split:',dict(images_c))
print('Byte-identical duplicates dropped:',duplicates)
print('Images missing labels (background candidates):',missing_annotations)
print('Labels:',len(class_names),'Original:',len(old_names),'New:',len(expanded))
print('New categories without any training boxes:',[x for x in expanded if boxes_c['train',x]==0])
assert not [x for x in expanded if boxes_c['train',x]==0],'Cannot train classes with no examples'
print('Few-shot new classes:',{x:boxes_c['train',x] for x in expanded if boxes_c['train',x]<25})
print('No new classes were silently omitted')
for part in SAM_SUBCLASSES:
    assert boxes_c['train',part]>0, 'Sam specialist class missing in unified training: '+part
assert class_names[:57]==old_names
assert len(class_names)==87

## 7 — Three progressive training stages

**Stage 1** builds the expanded 87-class head starting from Round 2. **Stage 2** refines the strongest checkpoint with less augmentation. **Stage 3** fine-tunes conservatively. Each uses the **same fixed validation split** to select the strongest checkpoint, and the separate test set remains untouched. This is not three independent accuracy multipliers: accuracy may improve, plateau or decline.

The notebook will take substantial GPU time, and Colab free-tier sessions can disconnect. If it runs out of GPU time, save interim checkpoints to Drive and resume, rather than rerunning everything blindly.

In [ ]:
# Train 3 STAGES with a fixed validation split, choose the strongest checkpoint.
# More epochs alone are NOT an accuracy guarantee. Only the final chosen
# checkpoint is evaluated on the untouched held-out test set.
from ultralytics import YOLO
import json
TRAIN_IMGSZ=640
TRAIN_BATCH=8
STAGES=[('foundation',35,.65),('refinement',25,.40),('fine_tune',15,.22)]
EXPERIMENTS=Path('/content/circuitvision_runs/unified_yolo11')
EXPERIMENTS.mkdir(parents=True,exist_ok=True)
MODEL_START=MODEL_PT  # Authentic Round 2 model; automatically adapts detection head to 87 classes.
BEST=None
best_score=float('-inf')
history=[]
old_critical=['Resistor','LED','LDR','Power Module','Diode','IC']
for stage_index,(stage_name,epochs,mosaic) in enumerate(STAGES,1):
    print('\nTraining phase',stage_index,'/',len(STAGES),stage_name,
          'from:',MODEL_START)
    m=YOLO(str(MODEL_START))
    m.train(data=str(MERGED/'data.yaml'),epochs=epochs,imgsz=TRAIN_IMGSZ,
            batch=TRAIN_BATCH,device=0,workers=2,patience=10,seed=47+stage_index,
            project=str(EXPERIMENTS),name=stage_name,exist_ok=True,
            amp=True,plots=True,close_mosaic=5,mosaic=mosaic,
            mixup=.03 if stage_index==1 else 0.,degrees=8,
            translate=.10,scale=.25 if stage_index==1 else .15,fliplr=.5)
    candidate=EXPERIMENTS/stage_name/'weights'/'best.pt'
    assert candidate.exists(),'Checkpoint missing for '+stage_name
    trained_stage=YOLO(str(candidate))
    assert [str(trained_stage.names[i]) for i in range(87)]==class_names, \
        'Class mapping changed unexpectedly during '+stage_name
    val=trained_stage.val(data=str(MERGED/'data.yaml'),split='val',
                          imgsz=TRAIN_IMGSZ,device=0,plots=False)
    ap={class_names[int(cid)]:float(val.box.ap50[k])
        for k,cid in enumerate(val.box.ap_class_index)}
    priority_vals=[ap[k] for k in old_critical if k in ap]
    specialist_vals=[ap[k] for k in SAM_SUBCLASSES if k in ap]
    priority_avg=sum(priority_vals)/len(priority_vals) if priority_vals else 0.
    specialist_avg=sum(specialist_vals)/len(specialist_vals) if specialist_vals else 0.
    score=.55*float(val.box.map50)+.30*priority_avg+.15*specialist_avg
    row=dict(stage=stage_name,epochs_requested=epochs,map50=float(val.box.map50),
             map50_95=float(val.box.map),priority_mean_ap50=priority_avg,
             specialist_mean_ap50=specialist_avg,validation_score=score,
             missing_specialist_validation=[k for k in SAM_SUBCLASSES if k not in ap])
    history.append(row)
    if score>best_score:
        BEST=candidate
        best_score=score
        print('New best validation checkpoint:',stage_name)
    else:
        print('No validation improvement; retaining previous better checkpoint')
    # Continue from the BEST available stage, not automatically the latest stage.
    MODEL_START=BEST
    print('Phase results:',json.dumps(row,indent=2))
assert BEST and BEST.exists()
print('\nSelected unified YOLO11 weights:',BEST)
print('Stages completed:',len(history),'best validation score:',best_score)
print('WARNING: Specialists with few holdout samples remain unverified.')

## 8 — Independent final evaluation

Only the selected **best-of-three validated checkpoint** is evaluated on the untouched test split. Measure old and new components separately. Sam's original notebook used a very small validation set; this new model cannot claim reliable exact-value readings without diverse, annotated and measured tests.

In [ ]:
# Evaluate selected winner ONLY ONCE on held-out test.
trained=YOLO(str(BEST))
assert [str(trained.names[i]) for i in range(len(class_names))]==class_names
metrics=trained.val(data=str(MERGED/'data.yaml'),split='test',imgsz=TRAIN_IMGSZ,
                    device=0,plots=False)
mAP50=float(metrics.box.map50);mAP5095=float(metrics.box.map)
per_ap={class_names[int(i)]:float(metrics.box.ap50[k])
        for k,i in enumerate(metrics.box.ap_class_index)}
print('Unified YOLO11 test mAP@50:',round(mAP50,4))
print('Unified YOLO11 test mAP@50–95:',round(mAP5095,4))
for label in ['Resistor','LED','LDR','Power Module','Diode','IC','Battery 9V',
              'ESP32','Oscilloscope','Multimeter','MOSFET']+SAM_SUBCLASSES:
    print(f'{label:22} test AP@50:',
          round(per_ap[label],3) if label in per_ap else 'No evaluable test examples')
print('Do NOT directly compare this to Round 2/Round 3 historical mAP scores:')
print('This is a new combined holdout and may contain tiny specialist tests.')
assert mAP50>0,'No valid test evaluation; review labels before exporting'

## 9 — Export exactly ONE physical-object model

The resulting ONNX file has 87 detector class channels. This replaces the need to run three models on each live video frame **if** the new detector proves accurate enough. It does NOT automatically replace your Render production checkpoint.

In [ ]:
import onnxruntime as ort
import hashlib,json,shutil,zipfile
onnx_file=Path(trained.export(format='onnx',imgsz=TRAIN_IMGSZ,opset=17,
                              dynamic=False,nms=False,simplify=True))
session=ort.InferenceSession(str(onnx_file),providers=['CPUExecutionProvider'])
shape=list(session.get_outputs()[0].shape)
assert (4+len(class_names)) in shape, 'Expected 4 + 87 ONNX detection channels'
assert list(session.get_inputs()[0].shape)==[1,3,640,640]
OUTPUT=Path('/content/CircuitVision_Unified_YOLO11')
OUTPUT.mkdir(exist_ok=True)
shutil.copy2(BEST,OUTPUT/'best.pt')
shutil.copy2(onnx_file,OUTPUT/'best.onnx')
(OUTPUT/'classes.json').write_text(json.dumps(
    {str(i):n for i,n in enumerate(class_names)},indent=2))
report=dict(version='CircuitVision unified YOLO11 candidate',class_count=len(class_names),
            original_round2_class_count=len(old_names),
            round3_additional_class_count=24,sam_specialist_subtypes=SAM_SUBCLASSES,
            sam_original_ten_class_map=SAM_MAP,three_training_stages=history,
            selected_checkpoint=str(BEST),test_map50=mAP50,test_map50_95=mAP5095,
            per_class_ap50=per_ap,sam_train_images=len(train),
            sam_val_images=len(valid),sam_test_images=len(test),
            image_counts={str(k):v for k,v in images_c.items()},
            near_duplicate_leakage_not_audited=True,
            claims_of_measured_resistor_values=False,
            warning='Model visual labels are NOT measured values. New test set and small Sam test make historical mAP comparison invalid. Never deploy without evaluation.')
(OUTPUT/'TRAINING_REPORT.json').write_text(json.dumps(report,indent=2))
(OUTPUT/'DATASET_SOURCES.txt').write_text(
    'Round 2 datasets: https://universe.roboflow.com/android-yolo/eesob, https://universe.roboflow.com/vishnupriyas-workspace/electronics-components-sveck\n'
    'Round 3 expansion: https://universe.roboflow.com/personal-8hq68/electronic-component-detection-m6sop, https://universe.roboflow.com/myspace-natcf/deteksi-alat-elektronik\n'
    'Sam source: student-provided private labeled data.zip; do NOT publish the original ZIP without explicit permission\n'
    'Model checkpoint starting weights: CircuitVision student-trained Round 2\n')
archive=Path('/content/CircuitVision_Unified_YOLO11_Candidate.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for f in sorted(OUTPUT.iterdir()):z.write(f,arcname=f.name)
print('Candidate ZIP:',archive,'size:',archive.stat().st_size)
print('SHA256:',hashlib.sha256(archive.read_bytes()).hexdigest())
print('Class count:',len(class_names),'training passes:',len(history))
print('Production remains unchanged; do NOT upload this to GitHub models/ yet.')

## 10 — Download the candidate and report

On your phone, the ZIP appears in Downloads if Chrome allows the transfer. **Upload the ZIP back in ChatGPT** before any deployment. The original Round 2 website stays untouched.

In [ ]:
from google.colab import files
files.download('/content/CircuitVision_Unified_YOLO11_Candidate.zip')

## Next steps: DO NOT REPLACE THE LIVE MODEL YET

Upload `CircuitVision_Unified_YOLO11_Candidate.zip` in this chat. Rad will check its classes, hashes, shape, per-class test scores and speed, then prepare a safe deployment with rollback. The present CircuitVision website remains on its existing stable model.

**Reality check:** The category list contains some overlapping labels (for example, 7 Seg vs 7-segment). Sam's resistor-value subclasses also overlap a generic Resistor class; we must benchmark those carefully, reject uncertain detail predictions, and collect harder test images if needed. Running the same data three times cannot correct weak or inconsistent annotations.